# Copy one schema's data

Moves rows for a single schema. **Not part of S1–S12** — the migration registers this notebook and never runs it. Moving data is a later decision the customer makes, with the notebook already sitting here.

One notebook per SCHEMA, never per table: a job run costs five to six minutes of startup, so per-table runs are the wrong shape.

---

*Generated from `engine/dataplane/02_copy_schema.py` by `engine/target/stage_notebooks.py`. Regenerate with `snowmig.py build-notebooks`; do not hand-edit — an edit here is overwritten on the next build. Change the source instead.*

In [ ]:
# ── PARAMETERS ─────────────────────────────────────────────────
# Edit these, then run the notebook top to bottom. Every value is a
# plain Python literal: `None` means the flag is not passed at all,
# and `True` means a bare switch is passed.
#
# Scope and mode are INPUTS. To migrate less, change a value here --
# never edit the stage logic below to make it cover less.
PARAMS = {
    'source-mode': 'connector',
    'source-config': None,
    'source-catalog': None,
    'target-catalog': None,  # REQUIRED
    'schema': None,  # REQUIRED
    'target-schema': None,
    'ddl-plan': None,
    'tables': None,
    'mode': 'skip-existing',
    'verify': 'counts',
    'reports-dir': None,
    'dry-run': False,
    'force': False,
}


def _argv(params, repeated=()):
    """PARAMS -> argv. None is omitted; True is a bare switch; a list
    follows its flag, or repeats the flag per value for a name in
    `repeated` (an append-style flag)."""
    argv = []
    for key, value in params.items():
        if value is None or value is False:
            continue
        if value is True:
            argv.append(f'--{key}')
            continue
        values = value if isinstance(value, (list, tuple)) else [value]
        if key in repeated:
            for v in values:
                argv.extend((f'--{key}', str(v)))
        else:
            argv.append(f'--{key}')
            argv.extend(str(v) for v in values)
    return argv


ARGV = _argv(PARAMS)
print('arguments:', ARGV)

_missing = [k for k in ['target-catalog', 'schema'] if not PARAMS.get(k)]
if _missing:
    raise ValueError(
        f'set these PARAMS before running: {_missing}')

## Shared source helpers

Inlined from `engine/dataplane/snowmig_source.py` so this notebook runs with nothing else uploaded beside it.

In [ ]:
"""How the migration scripts READ Snowflake from inside AIDP. Two modes.

LIVE-VERIFIED 2026-09-16 on a real cluster (Spark 3.5, AIDP 4.x):

  connector (default)  spark.read.format("aidataplatform") with
                       type=SNOWFLAKE. Talked to the account, read a table
                       (87 rows x 9 cols) and ran a pushdown query
                       (`current_user()` = the service user). Needs NO extra
                       cluster library — the format is built in — and needs
                       NO successful catalog crawl.

  external-catalog     three-part names `catalog.schema.table` against a
                       registered EXTERNAL catalog. Cheaper (no per-read
                       Snowflake login) but it can only see what the CRAWLER
                       has already discovered, and on the validated
                       deployment the crawler failed with
                       "CONNECTOR_0067 ... Login has timed out" while the
                       connector above worked with the same credentials.

So: connector mode is the default because it is the one proven end to end,
and external-catalog mode is kept for a deployment whose crawl succeeds.

The option NAMES are the live-verified raw ones, which differ from the
Python helper's keyword names — `user.name` (not `user`), `database.name`,
`authentication.method` ∈ {Basic, KeyPair}, `private.key.content`. A wrong
name fails loud (`DATA_ACCESS_LAYER_0001 - Required spark option ... was not
provided`), which is how these were established.

Credentials come from a CONFIG FILE, never from arguments: the same rule the
control plane follows. Point --source-config at a JSON file shaped like
snowmig-config.example.yaml (JSON, on the workspace mount).
"""
from __future__ import annotations

import json
import pathlib

__all__ = ["SOURCE_MODES", "SourceConfigError", "SnowflakeSource",
           "load_source_config"]

SOURCE_MODES = ("connector", "external-catalog")

AIDP_FORMAT = "aidataplatform"


# The verbs this transport may send. Deliberately narrower than the
# control-plane transport's list: WITH is absent, because following a CTE to
# the statement it prefixes needs the engine's lexer, which does not exist on
# a cluster. A read that needs a CTE can be written as a subquery.
PUSHDOWN_READ_VERBS = ("SELECT", "SHOW", "DESCRIBE", "DESC", "EXPLAIN")


class SourceWriteRefused(PermissionError):
    """A statement that is not a read was handed to the pushdown transport."""


def _code_only(sql: str) -> str:
    """`sql` with string literals and comments blanked, length preserved.

    A `;` inside a literal is data, not a statement boundary, and `--` or
    `//` inside one is not a comment. Blanking rather than deleting keeps offsets, so the
    scan cannot be confused about where anything starts.

    Lexed as SNOWFLAKE lexes: a backslash escapes only inside a '...'
    string; a "..." identifier ends at the first quote that is not doubled.
    Honouring a backslash there too made `"a\\"; delete from T` one
    identifier to this scan and two statements to Snowflake -- the guard
    failed open.
    """
    out = []
    i, n = 0, len(sql)
    while i < n:
        c = sql[i]
        two = sql[i:i + 2]
        if c in ("'", '"'):
            out.append(" ")
            i += 1
            while i < n:
                if c == "'" and sql[i] == "\\" and i + 1 < n:  # escape
                    out.append("  ")
                    i += 2
                    continue
                if sql[i] == c:
                    if sql[i:i + 2] == c * 2:           # doubled = literal
                        out.append("  ")
                        i += 2
                        continue
                    out.append(" ")
                    i += 1
                    break
                out.append("\n" if sql[i] == "\n" else " ")
                i += 1
            continue
        if two == "$$":
            out.append("  ")
            i += 2
            while i < n and sql[i:i + 2] != "$$":
                out.append("\n" if sql[i] == "\n" else " ")
                i += 1
            out.append("  ")
            i += 2
            continue
        # `//` is a line comment in Snowflake exactly as `--` is. Missing it
        # let an apostrophe in `// it's` open a phantom literal that hid the
        # `;` on the next line -- two statements read as one.
        if two in ("--", "//"):
            while i < n and sql[i] != "\n":
                out.append(" ")
                i += 1
            continue
        if two == "/*":
            depth, i = 1, i + 2
            out.append("  ")
            while i < n and depth:
                if sql[i:i + 2] == "/*":
                    depth += 1
                    out.append("  ")
                    i += 2
                elif sql[i:i + 2] == "*/":
                    depth -= 1
                    out.append("  ")
                    i += 2
                else:
                    out.append("\n" if sql[i] == "\n" else " ")
                    i += 1
            continue
        out.append(c)
        i += 1
    return "".join(out)


def assert_pushdown_read_only(sql: str) -> None:
    """Refuse anything that is not a single read. Fails closed.

    The cluster-side counterpart of the control plane's `assert_read_only`:
    the credential the notebook holds may well be able to write, and the
    only thing standing between a migration and a modified SOURCE is this
    check.
    """
    code = _code_only(sql or "")
    statements = [s for s in code.split(";") if s.strip()]
    if not statements:
        raise SourceWriteRefused(
            f"empty statement refused; this transport is read-only against "
            f"Snowflake (allowed: {', '.join(PUSHDOWN_READ_VERBS)})")
    if len(statements) > 1:
        raise SourceWriteRefused(
            f"{len(statements)} statements in one pushdown; refused. A "
            f"second statement is how a write rides along behind a read.")
    verb = statements[0].split()[0].upper() if statements[0].split() else ""
    if verb == "WITH":
        raise SourceWriteRefused(
            "a CTE is refused by this transport: deciding whether `WITH ... "
            "INSERT` is a read needs the engine's scanner, which does not "
            "run on the cluster. Write the CTE as a subquery.")
    if verb not in PUSHDOWN_READ_VERBS:
        raise SourceWriteRefused(
            f"{verb or 'unrecognised statement'} refused: this transport is "
            f"read-only against Snowflake, whatever the credential allows "
            f"(allowed: {', '.join(PUSHDOWN_READ_VERBS)}).")


class SourceConfigError(ValueError):
    """The source config is missing, unreadable or incomplete."""


def q(identifier: str) -> str:
    """Backtick-quote one Spark identifier."""
    return "`" + str(identifier).replace("`", "``") + "`"


def _sql_ident(identifier: str) -> str:
    """Double-quote one SNOWFLAKE identifier (the pushdown runs there)."""
    return '"' + str(identifier).replace('"', '""') + '"'


def _sql_literal(value: str) -> str:
    """Escape a string literal for Snowflake SQL.

    The backslash first: Snowflake reads it as an escape inside '...', so a
    table named `a\\` made `'a\\'` an unterminated literal that swallowed the
    SQL after it.
    """
    return str(value).replace("\\", "\\\\").replace("'", "''")


def load_source_config(path: str | pathlib.Path) -> dict:
    """Read the Snowflake connection config (JSON) from the workspace."""
    p = pathlib.Path(path).expanduser()
    try:
        text = p.read_text(encoding="utf-8")
    except OSError as exc:
        raise SourceConfigError(
            f"source config not readable at {p}: {exc.strerror}") from exc
    if p.suffix.lower() in (".yaml", ".yml"):
        try:
            import yaml
        except ImportError as exc:
            raise SourceConfigError(
                f"{p} is YAML but PyYAML is not on the cluster; write the "
                f"config as JSON instead") from exc
        data = yaml.safe_load(text) or {}
    else:
        data = json.loads(text) if text.strip() else {}
    if not isinstance(data, dict):
        raise SourceConfigError(f"{p}: expected a mapping at the top level")
    # THE MIGRATION CONFIG IS ONE FILE FOR BOTH ENDS: the Snowflake connection
    # nested under `snowflake:`, the AIDP coordinates under `aidp:`. That is
    # the file `provision --source-config` reads, and it uploads the
    # `snowflake:` block as JSON (`plan/<stem>.json`) -- so the shape that
    # reaches the mount is the nested one, and reading the top level for
    # `account` found nothing but the envelope key. The failure surfaced on
    # the cluster, as every required field missing at once, which reads
    # like a broken credential rather than a config one level too deep.
    nested = data.get("snowflake")
    if isinstance(nested, dict):
        return dict(nested)
    return data


class SnowflakeSource:
    """Read-only access to the Snowflake source, in either mode.

    Nothing here can write: every method issues a read, and the connector is
    read-only in AIDP 4.0 by Oracle's own statement.
    """

    def __init__(self, spark, *, mode: str = "connector",
                 config: dict | None = None,
                 external_catalog: str | None = None,
                 session_schema: str | None = None):
        if mode not in SOURCE_MODES:
            raise SourceConfigError(
                f"unknown source mode {mode!r}; expected one of "
                f"{list(SOURCE_MODES)}")
        self.spark = spark
        self.mode = mode
        self.external_catalog = external_catalog
        self._options: dict[str, str] = {}
        # A REAL schema, used only to scope a pushdown session. The connector
        # validates this option against the schemas it can see and rejects
        # INFORMATION_SCHEMA itself with DATA_ACCESS_LAYER_0031 -- but a
        # pushdown query scoped to a real schema may reference
        # INFORMATION_SCHEMA freely (both established live).
        self.session_schema = session_schema or (config or {}).get("schema")

        if mode == "external-catalog":
            if not external_catalog:
                raise SourceConfigError(
                    "external-catalog mode needs --source-catalog")
            return

        cfg = config or {}
        missing = [k for k in ("account", "warehouse", "database", "user",
                               "auth") if not cfg.get(k)]
        if missing:
            raise SourceConfigError(
                "source config is missing required field(s): "
                + ", ".join(sorted(missing)))
        account = str(cfg["account"]).strip()
        # The connector wants the account/server URL host, which is what the
        # Snowflake console calls "Account/Server URL".
        host = str(cfg.get("host")
                   or f"{account}.snowflakecomputing.com").strip()
        opts = {
            "type": "SNOWFLAKE",
            "host": host,
            "port": str(cfg.get("port") or 443),
            "database.name": str(cfg["database"]).strip(),
            "user.name": str(cfg["user"]).strip(),
            "warehouse": str(cfg["warehouse"]).strip(),
        }
        if cfg.get("role"):
            opts["role"] = str(cfg["role"]).strip()

        # A secret may be inline in the one config file, or in a file the
        # config points at. On a cluster the inline form is usually the only
        # one available, since the workspace mount carries the config but not
        # the operator's home directory.
        def secret(inline, path_field):
            if cfg.get(inline):
                return str(cfg[inline])
            if cfg.get(path_field):
                return pathlib.Path(
                    str(cfg[path_field])).expanduser().read_text(encoding="utf-8").strip()
            return None

        auth = str(cfg["auth"]).strip().lower()
        if auth == "keypair":
            key = secret("private_key", "key_path")
            if not key:
                raise SourceConfigError(
                    "auth: keypair needs `private_key` (inline) or `key_path`")
            opts["authentication.method"] = "KeyPair"
            opts["private.key.content"] = key
            passphrase = secret("key_passphrase", "key_passphrase_path")
            if passphrase:
                opts["private.key.pass.phrase"] = passphrase
        elif auth == "password":
            password = secret("password", "password_path")
            if not password:
                raise SourceConfigError(
                    "auth: password needs `password` (inline) or "
                    "`password_path`")
            opts["authentication.method"] = "Basic"
            opts["password"] = password
        else:
            raise SourceConfigError(
                f"auth {auth!r} is not supported by the AIDP Snowflake "
                f"connector; use keypair (preferred) or password")
        self._options = opts

    # -- describing the estate ------------------------------------------

    def database(self) -> str | None:
        return self._options.get("database.name")

    def pushdown(self, sql: str, *, schema: str | None = None):
        """Run `sql` IN SNOWFLAKE and return a DataFrame.

        Refuses anything that is not a single read statement, whatever the
        credential allows -- see `assert_pushdown_read_only`.

        Connector mode only. The `schema` option must name a REAL schema:
        the connector rejects INFORMATION_SCHEMA there with
        DATA_ACCESS_LAYER_0031, while a query scoped to a real schema may
        reference INFORMATION_SCHEMA freely. Both established live.
        """
        # Before the mode check and before anything reaches Spark: the
        # refusal must not depend on configuration being right.
        assert_pushdown_read_only(sql)
        if self.mode != "connector":
            raise SourceConfigError(
                "pushdown is connector-mode only; external-catalog mode has "
                "no Snowflake session to push into")
        scope = schema or self.session_schema
        if not scope:
            raise SourceConfigError(
                "connector pushdown needs a REAL schema to scope the "
                "session: add `schema:` to the source config or pass "
                "--session-schema. INFORMATION_SCHEMA is not accepted there.")
        return (self.spark.read.format(AIDP_FORMAT)
                .options(**self._options)
                .option("schema", scope)
                .option("pushdown.sql", sql)
                .load())

    def source_counts(self, schema: str, tables: list[str], *,
                      chunk: int = 50) -> dict[str, int]:
        """`{table: COUNT(*)}` for many tables in ONE round trip per chunk.

        Connector mode opens a Snowflake session per read, and the copy needs
        a source count twice per table (before, and again after, to catch a
        source that moved during the copy). Per-table counting therefore cost
        more session setup than the copy itself on a live run. A single
        UNION ALL answers a whole schema instead; it is chunked because a
        statement with thousands of branches is its own problem.

        External-catalog mode has no session to amortise, so it falls back to
        a count per table -- correct either way, and the caller does not care.
        """
        out: dict[str, int] = {}
        if self.mode != "connector":
            for table in tables:
                out[table] = self.read_table(schema, table).count()
            return out
        for start in range(0, len(tables), chunk):
            batch = tables[start:start + chunk]
            sql = " union all ".join(
                # The literal is the table's own name, so one query can carry
                # many counts and still say which is which. Both the literal
                # and the identifier are escaped: one apostrophe in a table
                # name would otherwise break the whole chunk.
                f"select '{_sql_literal(t)}' as SNOWMIG_TABLE, "
                f"count(*) as SNOWMIG_N from {_sql_ident(t)}"
                for t in batch)
            for row in self.pushdown(sql, schema=schema).collect():
                data = row.asDict()
                out[str(data["SNOWMIG_TABLE"])] = int(data["SNOWMIG_N"])
        return out

    def read_table(self, schema: str, table: str):
        """A DataFrame over one source table."""
        if self.mode == "external-catalog":
            return self.spark.table(
                f"{q(self.external_catalog)}.{q(schema)}.{q(table)}")
        return (self.spark.read.format(AIDP_FORMAT)
                .options(**self._options)
                .option("schema", schema)
                .option("table", table)
                .load())

    def register_temp_view(self, schema: str, table: str, view: str) -> str:
        """Expose a source table to SQL as a temp view, and return its name.

        INSERT ... SELECT needs the source addressable in SQL. In
        external-catalog mode the three-part name already is; in connector
        mode the DataFrame is registered as a session-local temp view, which
        is dropped by the caller.
        """
        if self.mode == "external-catalog":
            return f"{q(self.external_catalog)}.{q(schema)}.{q(table)}"
        self.read_table(schema, table).createOrReplaceTempView(view)
        return q(view)

    def drop_temp_view(self, view: str) -> None:
        if self.mode == "connector":
            self.spark.catalog.dropTempView(view)

    def describe(self) -> dict:
        """What this source is, for the report header. Never the credential."""
        out = {"mode": self.mode}
        if self.mode == "external-catalog":
            out["external_catalog"] = self.external_catalog
        else:
            out.update(session_schema=self.session_schema,
                       host=self._options.get("host"),
                       database=self._options.get("database.name"),
                       user=self._options.get("user.name"),
                       warehouse=self._options.get("warehouse"),
                       role=self._options.get("role"),
                       auth=self._options.get("authentication.method"))
        return out


## Stage logic

In [ ]:
#!/usr/bin/env python3
"""Copy ONE schema's tables from the external catalog into Delta, verified.

Runs on AIDP compute. Per table:

  1. read the source count;
  2. move the rows —
       skip-existing (default): only into a table with 0 rows; a table that
                                already holds rows is `skipped_nonempty`
                                when its count equals the source's and
                                `count_mismatch` when it does not -- a
                                re-run never softens a recorded failure;
       append:                  INSERT INTO ... SELECT <columns>;
       overwrite:               INSERT OVERWRITE ... SELECT <columns>
                                (rewrites ROWS, never drops the table);
     the source's columns are named, paired with the target's by name and
     listed in the target's order, so a source whose columns were reordered
     since the plan still lands each value in its own column;
  3. VERIFY: target count == source count (both read AFTER the copy), and
     with --verify counts+sums an exact SUM over every DECIMAL column OF
     THE SOURCE, cast to DECIMAL(38,s) with the SOURCE's scale on both
     sides. Floats are never summed for equality — float tolerance is
     wrong for money.

Before any row moves, and in both verify modes, the live source's columns
are checked against the target's BY NAME: a source column the target lacks,
or a target column the source lacks (renamed, dropped, added since the
plan), has no right place to land, so that table is recorded `type_drift`
and NOT copied. The source's DECIMAL columns are then checked against the
target's types: a target column that is not DECIMAL, or a DECIMAL with fewer
integer digits or a smaller scale, would be rounded or truncated by the
INSERT with the row count intact -- `type_drift` too, and NOT copied.

The copy's claim is the verification, not the INSERT returning: exactly the
discipline the control-plane deploy learned from live AIDP (a 2xx is not the
claim).

CONSISTENCY: each table is read at its own moment. If the source is still
being written, per-table counts can be exact and the SCHEMA still be
internally inconsistent. For a cutover: freeze writers or copy from a
point-in-time Snowflake CLONE. The report records copy timestamps so drift is
attributable.

Resumable: a table the report records as `verified` is skipped (--force
re-copies). Failures are recorded and the run continues; the report is the
deliverable.
"""
from __future__ import annotations

import argparse
import datetime
import json
import pathlib
import re
import sys
import time


# /Workspace is the live-verified mount of the workspace tree on cluster
# filesystems (probed 2026-09-16 on a real cluster).
DEFAULT_REPORTS_DIR = "/Workspace/backup-snowflake-migration/reports"
MANIFEST_NAME = "discovery_manifest.json"

_DECIMAL = re.compile(r"^decimal\((\d+)\s*,\s*(\d+)\)$", re.IGNORECASE)

# Copy statuses that mean the table is NOT verified. A later run that copies
# nothing (skip-existing over a table with rows) never softens one of these.
_COPY_FAILURES = ("count_mismatch", "sum_mismatch", "type_drift", "failed")

# Structure statuses that mean the table IS there. A copy that then cannot
# find it has not "nothing to do": it failed to copy into a created table.
_STRUCTURE_PRESENT = ("created", "already_existed")


def q(identifier: str) -> str:
    return "`" + str(identifier).replace("`", "``") + "`"


def three(*parts: str) -> str:
    return ".".join(q(p) for p in parts)


def log(msg: str) -> None:
    print(f"[copy] {msg}", flush=True)


def fail(msg: str) -> int:
    """Report a refusal on BOTH streams and return 1.

    A notebook task captures stdout only: live, a script that exited 1 via a
    stderr-only message produced a job failure with NO explanation anywhere.
    """
    print(f"ERROR: {msg}", flush=True)
    print(f"error: {msg}", file=sys.stderr)
    return 1


def _same(a: str, b: str) -> bool:
    """Spark resolves catalog and schema names case-insensitively, so two
    targets that differ only in case are the same place. Comparing them as
    exact strings dropped the structure report for `lake.core` when this run
    spelled it `lake.CORE`, and the copy fell back to the whole manifest."""
    return str(a).casefold() == str(b).casefold()


def planned_target_schemas(ddl_plan: dict, schema: str) -> set[str]:
    """Every target schema the approved plan puts source schema `schema` in.

    The same reading as 01_create_structure's `targets_from_ddl_plan`: a
    TABLE statement with a three-part `source_identifier` and a three-part
    `target_fqn`. 01 creates the table where the plan says, so the copy has
    to look there too -- deriving the schema from `--schema` again copied
    into `lake.CORE` while 01 had created `lake.db_core`.
    """
    out: set[str] = set()
    for stmt in ddl_plan.get("statements") or []:
        source = str(stmt.get("source_identifier") or "").split(".")
        target = str(stmt.get("target_fqn") or "").split(".")
        if len(source) != 3 or len(target) != 3:
            continue
        if str(stmt.get("object_type") or "TABLE").upper() == "VIEW":
            continue
        if source[1] == schema:
            out.add(target[1])
    return out


def planned_tables(ddl_plan: dict, schema: str,
                   target_schema: str) -> set[str]:
    """Casefolded source table names the plan puts at `target_schema`.

    The per-table reading of `planned_target_schemas`. `target_missing` for
    one of these is a failure whatever the structure report says: with
    `--tables` beside a report for another target, or before 01 ran at all,
    there is no report to say the table should be there -- and the copy
    exited 0 with 0 rows for a table the reviewed plan places here.
    """
    out: set[str] = set()
    for stmt in ddl_plan.get("statements") or []:
        source = str(stmt.get("source_identifier") or "").split(".")
        target = str(stmt.get("target_fqn") or "").split(".")
        if len(source) != 3 or len(target) != 3:
            continue
        if str(stmt.get("object_type") or "TABLE").upper() == "VIEW":
            continue
        if source[1] == schema and _same(target[1], target_schema):
            out.add(source[2].casefold())
    return out


def plan_catalogs(ddl_plan: dict) -> set[str]:
    """Every catalog the plan targets (01 refuses a run for another one)."""
    out = set()
    for stmt in ddl_plan.get("statements") or []:
        target = str(stmt.get("target_fqn") or "").split(".")
        if len(target) == 3:
            out.add(target[0])
    return out


def resolve_target_schema(schema: str, planned: set[str],
                          override: str | None) -> tuple[str | None, str | None]:
    """`(target_schema, None)`, or `(None, refusal)`: the rule 01 applies.

    The approved plan decides; `--target-schema` may restate it (in any
    case) but not contradict it; only where the plan is silent does the
    source schema name stand in.
    """
    if override:
        if not planned:
            return override, None
        match = next((p for p in sorted(planned) if _same(p, override)), None)
        if match is None:
            return None, (
                f"error: --target-schema {override!r} contradicts the "
                f"approved plan, which puts {schema} in "
                f"{', '.join(sorted(planned))} -- where 01_create_structure "
                f"created it. The plan is the reviewed artifact; change it, "
                f"or drop the flag.")
        return match, None
    if len(planned) == 1:
        return next(iter(planned)), None
    if len(planned) > 1:
        return None, (
            f"error: the approved plan puts source schema {schema} in more "
            f"than one target schema ({', '.join(sorted(planned))}); this "
            f"stage copies one schema per run. Pass --target-schema to say "
            f"which.")
    return schema, None


def _count(spark, fqn: str) -> int:
    return spark.sql(f"SELECT COUNT(*) AS n FROM {fqn}").collect()[0]["n"]


def _column_types(spark, fqn: str) -> dict[str, str]:
    """{column: data_type} from DESCRIBE, in column order; lower-cased types.

    Columns end at the first blank or `#` row (Delta's metadata section).
    """
    out: dict[str, str] = {}
    for row in spark.sql(f"DESCRIBE {fqn}").collect():
        name = str(row["col_name"] or "").strip()
        if not name or name.startswith("#"):
            break
        out[name] = str(row["data_type"] or "").strip().lower()
    return out


def _decimal_columns(types: dict[str, str]) -> list[tuple[str, int, int]]:
    """[(column, precision, scale)] for every DECIMAL column in `types`."""
    out = []
    for name, data_type in types.items():
        m = _DECIMAL.match(data_type)
        if m:
            out.append((name, int(m.group(1)), int(m.group(2))))
    return out


def _layout_drift(src_types: dict[str, str],
                  tgt_types: dict[str, str]) -> tuple[dict, str] | None:
    """`(drift, why)` when the source and target columns are not the same
    NAMES, else None. Case-insensitive, as Spark resolves column names.

    The target was checked against the plan by the structure step; nothing
    checked it against the LIVE source, which can have been rebuilt since.
    Column ORDER is not drift: the INSERT names every column and pairs them
    by name. A name that is on one side only is, since its values have no
    right place to land -- positionally, an email ended up in `city`.
    """
    for side, types in (("source", src_types), ("target", tgt_types)):
        folded: dict[str, list[str]] = {}
        for name in types:
            folded.setdefault(name.casefold(), []).append(name)
        clash = [names for names in folded.values() if len(names) > 1]
        if clash:
            return ({f"{side}_names_differing_only_in_case": clash[0]},
                    f"the {side} has columns whose names differ only in "
                    f"case ({', '.join(clash[0])}); Spark resolves them as "
                    f"one name, so which value lands where cannot be "
                    f"decided")
    src = {n.casefold() for n in src_types}
    tgt = {n.casefold() for n in tgt_types}
    not_on_target = [n for n in src_types if n.casefold() not in tgt]
    not_in_source = [n for n in tgt_types if n.casefold() not in src]
    if not (not_on_target or not_in_source):
        return None
    parts = []
    if not_on_target:
        parts.append(f"source column(s) {', '.join(not_on_target)} are not "
                     f"on the target")
    if not_in_source:
        parts.append(f"target column(s) {', '.join(not_in_source)} are not "
                     f"in the source")
    return ({"not_on_target": not_on_target, "not_in_source": not_in_source},
            "; ".join(parts) + " -- renamed, dropped or added since the plan")


def _column_pairs(src_types: dict[str, str],
                  tgt_types: dict[str, str]) -> list[tuple[str, str]]:
    """`[(target_column, source_column)]` in the TARGET's order, paired by
    name. Only called once `_layout_drift` found the same names both sides."""
    by_fold = {n.casefold(): n for n in src_types}
    return [(t, by_fold[t.casefold()]) for t in tgt_types]


def _type_drift(src_types: dict[str, str], tgt_types: dict[str, str]) -> dict:
    """Source DECIMAL columns the target cannot hold without silent loss.

    Keyed off the SOURCE: a source decimal whose target column is not a
    decimal, or a decimal with fewer integer digits (precision - scale) or a
    smaller scale, would be rounded, truncated or overflowed by the INSERT's
    store-assignment cast -- with the row count intact. A wider target is
    fine. Non-decimal columns are the structure stage's business.
    """
    by_lower = {k.lower(): v for k, v in tgt_types.items()}
    drift = {}
    for name, precision, scale in _decimal_columns(src_types):
        target = by_lower.get(name.lower())
        m = _DECIMAL.match(target or "")
        if not m or int(m.group(2)) < scale or \
                int(m.group(1)) - int(m.group(2)) < precision - scale:
            drift[name] = {"source": src_types[name],
                           "target": target or "<missing>"}
    return drift


def _decimal_sums(spark, fqn: str, columns: list[tuple[str, int]]) -> dict:
    if not columns:
        return {}
    selects = ", ".join(
        f"CAST(SUM(CAST({q(c)} AS DECIMAL(38,{s}))) AS STRING) AS {q(c)}"
        for c, s in columns)
    row = spark.sql(f"SELECT {selects} FROM {fqn}").collect()[0].asDict()
    return {k: row[k] for k in row}


# What Spark says when a table, or the schema holding it, is simply not
# there. Only these mean "absent".
_NOT_FOUND = ("TABLE_OR_VIEW_NOT_FOUND", "SCHEMA_NOT_FOUND",
              "NoSuchTableException", "NoSuchNamespaceException",
              "NoSuchDatabaseException", "Table or view not found")


def _target_exists(spark, tgt: str) -> bool:
    """True when DESCRIBE works, False when Spark says it is not there.

    Any OTHER error propagates. Every DESCRIBE error used to read as
    "absent": a metastore timeout or a persistent INSUFFICIENT_PERMISSIONS
    on a table 01 had just created became `target_missing` with the error
    thrown away -- "could not look" recorded as "not there", on every re-run.
    """
    try:
        spark.sql(f"DESCRIBE {tgt}")
        return True
    except Exception as exc:
        text = str(exc)
        if any(marker.lower() in text.lower() for marker in _NOT_FOUND):
            return False
        raise


def copy_table(source, schema: str, table: str, tgt: str, *, mode: str,
               verify: str, retries: int = 2, retry_wait: float = 30.0,
               source_count: int | None = None) -> dict:
    """Copy ONE table and verify it. The source is addressed through
    `SnowflakeSource`, so connector mode (a temp view over the connector
    read) and external-catalog mode (a three-part name) share this path."""
    spark = source.spark
    started = datetime.datetime.now(datetime.timezone.utc).isoformat()

    # A table with no target is a FINDING, not a crash. Live, the copy died
    # on the sixth table of a schema because the approved plan covered five
    # and the manifest listed a thousand -- taking the whole run with it.
    try:
        exists = _target_exists(spark, tgt)
    except Exception as exc:
        return {"status": "failed", "started_at": started,
                "reason": f"could not DESCRIBE {tgt}: {str(exc)[:300]}. "
                          f"Whether it exists is UNKNOWN, so nothing was "
                          f"copied. NOT verified."}
    if not exists:
        return {"status": "target_missing", "started_at": started,
                "reason": f"{tgt} does not exist, so there is nothing to copy "
                          f"into. Most often the table is not in the approved "
                          f"plan (structure reports it `not_in_plan`); run "
                          f"01_create_structure for it first if it should be."}

    view = f"snowmig_src_{schema}_{table}".lower()[:120]
    src = source.register_temp_view(schema, table, view)
    try:
        return _copy(spark, src, tgt, mode=mode, verify=verify,
                     retries=retries, retry_wait=retry_wait, started=started,
                     source_count=source_count)
    finally:
        source.drop_temp_view(view)


def _copy(spark, src: str, tgt: str, *, mode: str, verify: str,
          retries: int, retry_wait: float, started: str,
          source_count: int | None = None) -> dict:
    # The batched count from the caller when there is one: a per-table
    # COUNT(*) opens its own Snowflake session in connector mode.
    if source_count is None:
        source_count = _count(spark, src)

    # Pre-flight, before anything is written and in both verify modes:
    # metadata only (DESCRIBE on the registered source and on the target).
    src_types = _column_types(spark, src)
    tgt_types = _column_types(spark, tgt)
    if not src_types or not tgt_types:
        # Nothing to pair by name: could not look, not a match.
        return {"status": "failed", "source_count": source_count,
                "started_at": started,
                "reason": f"DESCRIBE of the "
                          f"{'source' if not src_types else 'target'} "
                          f"returned no columns, so its layout cannot be "
                          f"compared with the other side's. NOT copied."}
    layout = _layout_drift(src_types, tgt_types)
    if layout:
        drift, why = layout
        return {"status": "type_drift", "layout_drift": drift,
                "source_count": source_count, "started_at": started,
                "reason": f"{why}. The source's columns are not the "
                          f"target's, so its rows have no right place to "
                          f"land. NOT copied. Re-plan the table from the "
                          f"live source, or restore the source's layout."}
    drift = _type_drift(src_types, tgt_types)
    if drift:
        return {"status": "type_drift", "type_drift": drift,
                "source_count": source_count, "started_at": started,
                "reason": f"{len(drift)} DECIMAL column(s) are narrower or "
                          f"not DECIMAL on the target; an INSERT would round "
                          f"or truncate them silently. NOT copied. Recreate "
                          f"the table from the approved plan."}

    target_rows = _count(spark, tgt)
    if mode == "skip-existing" and target_rows > 0:
        # A verification, not a bypass: a target that holds rows but not the
        # source's count is a mismatch whether or not this run wrote it.
        if target_rows != source_count:
            return {"status": "count_mismatch", "source_count": source_count,
                    "target_count": target_rows, "started_at": started,
                    "reason": f"target already holds {target_rows} row(s) "
                              f"but the source has {source_count}; nothing "
                              f"was copied. Use --mode overwrite to rewrite "
                              f"it. NOT verified."}
        return {"status": "skipped_nonempty", "source_count": source_count,
                "target_count": target_rows, "started_at": started,
                "reason": f"target already holds {target_rows} row(s); use "
                          f"--mode overwrite to rewrite them or append to add"}

    # Every source column named, in the TARGET's order: an INSERT fills the
    # target positionally, and `SELECT *` is the SOURCE's order, so a source
    # rebuilt with two columns swapped landed each value in the other's
    # column with the counts intact. (No target column list: the select
    # list already is the target's order, and it keeps the statement the
    # plain INSERT ... SELECT every Delta version accepts.)
    select = ", ".join(q(s) for _t, s in _column_pairs(src_types, tgt_types))
    verb = "INSERT OVERWRITE" if mode == "overwrite" else "INSERT INTO"
    statement = f"{verb} {tgt} SELECT {select} FROM {src}"
    last_error = None
    for attempt in range(retries + 1):
        try:
            spark.sql(statement)
            last_error = None
            break
        except Exception as exc:
            last_error = str(exc)[:400]
            if attempt < retries:
                log(f"  retry {attempt + 1}/{retries} after error: "
                    f"{last_error[:120]}")
                time.sleep(retry_wait)
    if last_error is not None:
        return {"status": "failed", "source_count": source_count,
                "started_at": started, "reason": last_error}

    out = {"started_at": started,
           "finished_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
           "mode": mode}

    # The verification IS the claim. The rows have landed by now, so a
    # failure from here on must say so: a `failed` record that looked like a
    # failed INSERT would invite an `append` re-run that duplicates every row.
    try:
        return _verify(spark, src, tgt, out, verify=verify,
                       source_count=source_count, src_types=src_types)
    except Exception as exc:
        out.update(status="failed", insert_completed=True,
                   reason=f"the INSERT completed but the verification raised: "
                          f"{str(exc)[:300]}. NOT verified; re-copy with "
                          f"--mode overwrite, not append")
        return out


def _verify(spark, src: str, tgt: str, out: dict, *, verify: str,
            source_count: int, src_types: dict[str, str]) -> dict:
    src_after = _count(spark, src)
    tgt_after = _count(spark, tgt)
    out.update(source_count=src_after, target_count=tgt_after)
    if src_after != source_count:
        out["source_moved_during_copy"] = (
            f"source count changed {source_count} -> {src_after} during the "
            f"copy; the source is still being written")
    if tgt_after != src_after:
        out["status"] = "count_mismatch"
        out["reason"] = (f"target has {tgt_after} row(s), source has "
                         f"{src_after}. NOT verified.")
        return out

    if verify == "counts+sums":
        # The SOURCE's decimal columns, cast to the SOURCE's scale on both
        # sides: the target is at least as wide (the pre-flight in _copy
        # refused it otherwise), so the comparison is exact rather than
        # rounded to whatever the target happens to be.
        columns = [(c, s) for c, _p, s in _decimal_columns(src_types)]
        src_sums = _decimal_sums(spark, src, columns)
        tgt_sums = _decimal_sums(spark, tgt, columns)
        sum_drift = {c: {"source": src_sums[c], "target": tgt_sums[c]}
                     for c, _ in columns if src_sums[c] != tgt_sums[c]}
        out["decimal_columns_checked"] = [c for c, _ in columns]
        if not columns:
            out["decimal_columns_note"] = ("the source has no DECIMAL "
                                           "columns; counts are the whole check")
        if sum_drift:
            out["status"] = "sum_mismatch"
            out["sum_drift"] = sum_drift
            out["reason"] = (f"{len(sum_drift)} decimal column(s) do not sum "
                             f"equal. NOT verified.")
            return out

    out["status"] = "verified"
    return out


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--source-mode", choices=list(SOURCE_MODES),
                    default="connector",
                    help="how to READ the source (see snowmig_source.py). "
                         "connector is the live-verified default and needs "
                         "no successful catalog crawl")
    ap.add_argument("--source-config",
                    help="JSON/YAML connection config (connector mode)")
    ap.add_argument("--source-catalog",
                    help="the registered EXTERNAL catalog "
                         "(external-catalog mode)")
    ap.add_argument("--target-catalog", required=True)
    ap.add_argument("--schema", required=True,
                    help="ONE schema per run — that is the operating unit")
    ap.add_argument("--target-schema", default=None,
                    help="default: the schema the approved plan's "
                         "target_fqn names for --schema, as in "
                         "01_create_structure; --schema itself only where "
                         "the plan is silent. Refused when it contradicts "
                         "the plan")
    ap.add_argument("--ddl-plan",
                    help="path to ddl_plan.json (default: ../plan/"
                         "ddl_plan.json next to --reports-dir); read for "
                         "the target schema only")
    ap.add_argument("--tables", nargs="*", default=None,
                    help="subset; default: every table the manifest lists")
    ap.add_argument("--mode", choices=("skip-existing", "append", "overwrite"),
                    default="skip-existing")
    ap.add_argument("--verify", choices=("counts", "counts+sums"),
                    default="counts")
    ap.add_argument("--reports-dir", default=DEFAULT_REPORTS_DIR)
    ap.add_argument("--dry-run", action="store_true")
    ap.add_argument("--force", action="store_true",
                    help="re-copy tables already recorded as verified; needs "
                         "--mode overwrite or append, since skip-existing "
                         "cannot re-copy a table that holds rows")
    args = ap.parse_args(argv)

    if args.source_catalog and \
            args.source_catalog.lower() == args.target_catalog.lower():
        return fail("error: source and target catalog are the same.")
    if args.force and args.mode == "skip-existing":
        # Under the default mode --force copied nothing (skip-existing never
        # writes into a table with rows) and overwrote a `verified` record
        # with `skipped_nonempty`. Refuse rather than guess at overwrite.
        return fail("error: --force re-copies tables already recorded as "
                    "verified, which --mode skip-existing cannot do; pass "
                    "--mode overwrite (rewrites rows) or --mode append")

    reports = pathlib.Path(args.reports_dir)
    manifest = json.loads((reports / MANIFEST_NAME).read_text(encoding="utf-8"))
    record = next((s for s in manifest["schemas"] if s["name"] == args.schema),
                  None)
    if record is None:
        return fail(f"error: schema {args.schema!r} not in the manifest")

    # The approved plan decides the target namespace, exactly as it does for
    # 01_create_structure. A plan that is not there is a silent plan (01 in
    # --mode ctas or manifest never reads one); a plan the operator NAMED
    # and that is not there is a mistake worth stopping on.
    ddl_path = (pathlib.Path(args.ddl_plan) if args.ddl_plan
                else reports.parent / "plan" / "ddl_plan.json")
    planned: set[str] = set()
    ddl_plan = None
    if ddl_path.is_file():
        ddl_plan = json.loads(ddl_path.read_text(encoding="utf-8"))
        stray = {c for c in plan_catalogs(ddl_plan)
                 if not _same(c, args.target_catalog)}
        if stray:
            return fail(
                f"error: the approved plan targets catalog(s) "
                f"{', '.join(sorted(stray))}, and this run was given "
                f"--target-catalog {args.target_catalog}; "
                f"01_create_structure refuses that pair, so there is "
                f"nothing here it created. Point this run at the catalog "
                f"the plan names.")
        planned = planned_target_schemas(ddl_plan, args.schema)
    elif args.ddl_plan:
        return fail(f"error: --ddl-plan {ddl_path} is not there")
    target_schema, refusal = resolve_target_schema(
        args.schema, planned, args.target_schema)
    if refusal:
        return fail(refusal)
    log(f"target schema {target_schema}: "
        + ("from the approved plan" if planned else
           "from --target-schema" if args.target_schema else
           "the source schema's own name (the plan names none for it)"))

    path = reports / f"copy_report_{args.schema.lower()}.json"
    target = f"{args.target_catalog}.{target_schema}"
    in_plan = (planned_tables(ddl_plan, args.schema, target_schema)
               if ddl_plan else set())

    # What the structure step recorded for THIS target. A report for another
    # target is not evidence about this one -- and falling back to the whole
    # manifest because of it is how a drifted table, excluded there, came
    # back into the copy's scope. Refused unless --tables names the scope.
    structure_path = reports / f"structure_report_{args.schema.lower()}.json"
    objects = None
    if structure_path.is_file():
        s_prior = json.loads(structure_path.read_text(encoding="utf-8"))
        s_target = s_prior.get("target")
        if s_target and not _same(s_target, target):
            if not args.tables:
                # --target-schema is a way out only where the plan is
                # silent; where it names a target, that flag is refused as a
                # contradiction, so offering it pointed at a dead end.
                way_out = ("" if planned else
                           f"pass --target-schema "
                           f"{s_target.split('.', 1)[-1]} (the plan names no "
                           f"target for this schema), ")
                return fail(
                    f"error: the structure report for {args.schema} targets "
                    f"{s_target}, and this copy resolves {target}. Taking "
                    f"the scope from the manifest instead would copy into "
                    f"tables the structure step never created or checked "
                    f"there. Re-run 01_create_structure (it creates what "
                    f"the plan names), {way_out}or pass --tables")
            log(f"the structure report targets {s_target}, not {target}; "
                f"not used for this run (--tables sets the scope)")
        else:
            objects = s_prior.get("objects") or {}

    report = {"schema": args.schema, "tables": {}, "target": target}
    if path.exists():
        prior = json.loads(path.read_text(encoding="utf-8"))
        # Resumability is keyed by SOURCE schema, so a report written against
        # a DIFFERENT target must not let this run skip copies as already
        # verified (the same trap the structure script hit live). Compared
        # case-insensitively: `lake.CORE` and `lake.core` are one schema.
        if prior.get("target") and not _same(prior["target"], target):
            log(f"the previous report targeted {prior['target']}, not "
                f"{target} — starting a fresh record for this target")
            path.with_suffix(
                f".{prior['target'].replace('.', '_')}.json").write_text(
                    json.dumps(prior, indent=2), encoding="utf-8")
        else:
            report = prior
            report["target"] = target

    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

    try:
        config = (load_source_config(args.source_config)
                  if args.source_config else None)
        source = SnowflakeSource(spark, mode=args.source_mode, config=config,
                                 external_catalog=args.source_catalog)
    except SourceConfigError as exc:
        return fail(f"error: {exc}")
    report["source"] = source.describe()

    if args.tables:
        names = list(args.tables)
    else:
        # Default to what the structure step created for THIS target, when it
        # left a report: the manifest is the whole estate, and copying into
        # tables nobody approved is not a default worth having. A table it
        # found already there WITH the planned layout counts; one it recorded
        # as `type_drift` never does -- the copy below is a positional INSERT
        # INTO ... SELECT *, and that layout is not the plan's.
        created = [n for n, rec in (objects or {}).items()
                   if rec.get("status") in _STRUCTURE_PRESENT]
        if created:
            names = created
            log(f"scope: {len(names)} table(s) the structure step created for "
                f"{target} (the manifest lists "
                f'{len(record["tables"])} for this schema)')
        else:
            # The `created` filter above never runs when nothing was created,
            # and that is exactly the all-drift schema: a re-plan over tables
            # that all pre-exist with the old layout records every one of
            # them `type_drift`. They are excluded here too, or the fallback
            # copies into the very layout the structure step refused.
            drifted = {n for n, r in (objects or {}).items()
                       if r.get("status") == "type_drift"}
            names = [t["name"] for t in record["tables"]
                     if t["name"] not in drifted]
            if objects is None:
                why = f"no structure report for {target} was found"
            else:
                # The report IS there; saying it was not pointed the operator
                # away from the real cause (a plan that never covered this
                # schema). Tables it created nothing for will come back
                # `target_missing` below.
                nip = sum(1 for r in objects.values()
                          if r.get("status") == "not_in_plan")
                why = (f"the structure report for {target} records 0 created "
                       f"table(s) ({nip} not_in_plan, {len(drifted)} "
                       f"type_drift, {len(objects) - nip - len(drifted)} "
                       f"other) -- re-run 01_create_structure with the right "
                       f"ddl_plan.json, or pass --tables")
                if drifted:
                    why += (f"; {len(drifted)} type_drift table(s) excluded "
                            f"-- recreate them from the approved plan")
                if drifted and not names:
                    # Zero iterations below would be exit 0: a copy job that
                    # did nothing, reported as a success.
                    return fail(
                        f"error: every table the manifest lists for "
                        f"{args.schema} is recorded type_drift in the "
                        f"structure report for {target}; nothing to copy. "
                        f"Recreate them from the approved plan "
                        f"(01_create_structure) or pass --tables to override")
            log(f"scope: all {len(names)} table(s) the manifest lists for "
                f"this schema; {why}")

    todo = [n for n in names
            if args.force
            or report["tables"].get(n, {}).get("status") != "verified"]

    # ONE round trip for every source count in this schema, rather than one
    # per table: session setup dominated the copy on a live run.
    source_counts: dict = {}
    if todo and not args.dry_run:
        try:
            source_counts = source.source_counts(args.schema, todo)
            log(f"source counts for {len(source_counts)} table(s) in "
                f"{(len(todo) + 49) // 50} round trip(s)")
        except Exception as exc:
            log(f"batched source counts unavailable ({str(exc)[:120]}); "
                f"falling back to one count per table")

    failures = 0
    for name in names:
        prior = report["tables"].get(name, {})
        if prior.get("status") == "verified" and not args.force:
            log(f"skip {args.schema}.{name}: already verified")
            continue
        tgt = three(args.target_catalog, target_schema, name)
        if args.dry_run:
            log(f"DRY RUN: would copy {args.schema}.{name} -> {tgt} "
                f"({args.mode}, verify={args.verify}, "
                f"source={args.source_mode})")
            continue
        log(f"{args.schema}.{name}: copying ({args.mode})")
        started = datetime.datetime.now(datetime.timezone.utc).isoformat()
        try:
            result = copy_table(source, args.schema, name, tgt, mode=args.mode,
                                verify=args.verify,
                                source_count=source_counts.get(name))
        except Exception as exc:
            # A failure is a finding, not the end of the run: live, one
            # connector login timeout would otherwise end the schema with
            # the failing table unrecorded and the rest never attempted.
            result = {"status": "failed", "started_at": started,
                      "reason": str(exc)[:400]}
            log(f"{args.schema}.{name}: FAILED — {str(exc)[:200]}")
        if result["status"] == "skipped_nonempty" and \
                prior.get("status") in _COPY_FAILURES:
            # Nothing was copied, so nothing was re-verified: a recorded
            # failure stands until a real re-copy verifies the table.
            result = dict(prior, reason=(
                f"{prior.get('reason') or prior['status']} (a re-run in "
                f"skip-existing mode left the target untouched; use --mode "
                f"overwrite to re-copy and re-verify it)"))
        # `target_missing` is a finding for a table the structure step never
        # created (not in the plan). For one it records as there, or one the
        # approved plan places at this target, the copy moved nothing into a
        # table that should exist: a failure, or the job reads SUCCESS with
        # 0 rows copied. The plan counts on its own: `--tables` beside a
        # report for another target, or a copy run before 01, has no report
        # for this target to say so.
        s_status = (objects or {}).get(name, {}).get("status")
        if result["status"] == "target_missing" and \
                s_status in _STRUCTURE_PRESENT:
            result["reason"] = (
                f"the structure report records this table `{s_status}` in "
                f"{target}, yet {tgt} is not there now -- dropped since, or "
                f"created somewhere else. NOT copied.")
            failures += 1
        elif result["status"] == "target_missing" and \
                name.casefold() in in_plan:
            result["reason"] = (
                f"the approved plan places this table in {target}, yet {tgt} "
                f"is not there -- 01_create_structure has not created it "
                f"there (run it first), or it was dropped since. NOT copied.")
            failures += 1
        elif result["status"] not in ("verified", "skipped_nonempty",
                                      "target_missing"):
            failures += 1
        report["tables"][name] = result
        report["updated_at"] = datetime.datetime.now(
            datetime.timezone.utc).isoformat()
        path.write_text(json.dumps(report, indent=2), encoding="utf-8")
        log(f"{args.schema}.{name}: {result['status']} "
            f"({result.get('target_count', '?')} row(s))")

    statuses = {}
    for t in report["tables"].values():
        statuses[t["status"]] = statuses.get(t["status"], 0) + 1
    log(f"{args.schema}: {statuses} -> {path}")
    return 1 if failures else 0



In [ ]:
# ── RUN ────────────────────────────────────────────────────────────
# main() RETURNS an exit code; it is not allowed to raise SystemExit here.
# A notebook cell that raises SystemExit is reported as a FAILED task even
# when the work succeeded, so the code is inspected and only a real failure
# is re-raised -- which keeps a genuinely failed stage failing.
code = main(ARGV)
print('exit code:', code, flush=True)
if code:
    raise RuntimeError(f'stage exited {code}')
